## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:杨鑫叶


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

#define fi first
#define se second
#define pb push_back

typedef long long LL;

const int MAXN = 1024 + 5;

// 全局变量
int n, A, B, lowbitLen;
int p[MAXN], pos[MAXN];
vector<int> answer;

// ------------------------
// 工具函数
// ------------------------
inline int readInt() {
    int x = 0;
    char ch = getchar();
    while (ch < '0' || ch > '9') ch = getchar();
    while (ch >= '0' && ch <= '9') {
        x = x * 10 + (ch - '0');
        ch = getchar();
    }
    return x;
}

// 重建位置数组
inline void rebuildPos() {
    for (int i = 0; i < n; ++i) pos[p[i]] = i;
}

// 使用交换魔法
inline void useSwapMagic() {
    answer.pb(0);
    for (int i = 0; i < n; ++i) {
        if (p[i] == A) p[i] = B;
        else if (p[i] == B) p[i] = A;
    }
    rebuildPos();
}

// 使用加法魔法
inline void useAddMagic(int v) {
    v = (v % n + n) % n;
    if (v == 0) return;
    answer.pb(v);
    for (int i = 0; i < n; ++i) p[i] = (p[i] + v) % n;
    rebuildPos();
}

// 使用异或魔法
inline void useXorMagic(int v) {
    if (v == 0) return;
    answer.pb(-v);
    for (int i = 0; i < n; ++i) p[i] ^= v;
    rebuildPos();
}

// 计算对应的配对位置
inline void getPairPos(int x, int y, int &px, int &py) {
    int delta = (y - x + n - lowbitLen + n) % n;
    px = py = 0;
    for (int step = n / 2; step >= 2 * lowbitLen; step >>= 1) {
        if (delta >= step) {
            delta -= step;
            py += step / 2;
        } else {
            px += step / 2;
        }
    }
    px += n / 2 + (x & (lowbitLen - 1));
    py += (x & (lowbitLen - 1));
}

// 对两个位置进行递归交换
void applySwap(int x, int y) {
    if ((x / lowbitLen) % 2 == (y / lowbitLen) % 2) {
        int mid = ((x / lowbitLen) % 2 == 0) ? (x & (lowbitLen - 1)) + lowbitLen : (x & (lowbitLen - 1));
        applySwap(x, mid);
        applySwap(y, mid);
        applySwap(x, mid);
        return;
    }

    int pA, pB, pX, pY;
    getPairPos(A, B, pA, pB);
    getPairPos(x, y, pX, pY);

    useAddMagic((pX - x + n) % n);
    useXorMagic(pX ^ pA);
    useAddMagic((A - pA + n) % n);

    useSwapMagic();

    useAddMagic((pA - A + n) % n);
    useXorMagic(pX ^ pA);
    useAddMagic((x - pX + n) % n);
}

// ------------------------
// Permutation 类
// ------------------------
struct Permutation {
    int val[MAXN];
    int len;
    vector<int> ops;

    bool operator < (const Permutation &other) const {
        for (int i = 0; i < len; ++i)
            if (val[i] != other.val[i]) return val[i] < other.val[i];
        return false;
    }

    bool checkSorted() const {
        for (int i = 0; i + 1 < len; ++i)
            if (val[i] > val[i + 1]) return false;
        return true;
    }

    Permutation inverse() const {
        Permutation res;
        res.len = len;
        for (int i = 0; i < len; ++i) res.val[val[i]] = i;
        return res;
    }

    bool build() {
        bool vis[100005] = {};
        for (int i = 0; i < len; ++i) vis[i] = true;
        for (int i = 0; i < len; ++i) if (!vis[i]) return false;
        if (len == 1) return true;

        Permutation leftPart, rightPart;
        leftPart.len = rightPart.len = len / 2;

        for (int i = 0; i < len / 2; ++i) {
            leftPart.val[i] = val[i * 2] / 2;
            rightPart.val[i] = val[i * 2 + 1] / 2;
        }

        if (!leftPart.build() || !rightPart.build()) return false;
        if (val[0] & 1) ops.pb(len == 2 ? 1 : -1);

        int curXorL = 0;
        for (int x : leftPart.ops) {
            if (x > 0) { ops.pb(-1); ops.pb(1); } 
            else { ops.pb(x * 2); curXorL ^= (-x) * 2; }
        }
        if (curXorL) ops.pb(-curXorL);

        int curXorR = 0;
        for (int x : rightPart.ops) {
            if (x > 0) { ops.pb(1); ops.pb(-1); } 
            else { ops.pb(x * 2); curXorR ^= (-x) * 2; }
        }

        if ((curXorR & (len / 2)) != (curXorL & (len / 2))) return false;
        if (curXorL >= len / 2) curXorL -= len / 2;
        if (curXorR >= len / 2) curXorR -= len / 2;
        if (curXorL != curXorR) return false;

        vector<int> merged;
        for (int x : ops) {
            if (merged.empty()) merged.pb(x);
            else {
                if (x < 0 && merged.back() < 0) {
                    merged.back() = -((-merged.back()) ^ (-x));
                    if (merged.back() == 0) merged.pop_back();
                } else merged.pb(x);
            }
        }
        ops.swap(merged);
        return true;
    }
};

// ------------------------
// 主函数
// ------------------------
int main() {
    n = readInt();
    A = readInt();
    B = readInt();

    for (int i = 0; i < n; ++i) p[i] = readInt();
    rebuildPos();

    lowbitLen = (A - B + n) % n;
    lowbitLen &= -lowbitLen;
    if (lowbitLen == 0) lowbitLen = n;

    if (lowbitLen > 1) {
        Permutation base;
        base.len = lowbitLen;
        for (int i = 0; i < n; ++i) base.val[i] = p[i] & (lowbitLen - 1);
        if (!base.build()) { printf("-1\n"); return 0; }

        for (int x : base.ops) {
            if (x > 0) useAddMagic(x); else useXorMagic(-x);
        }
    }

    for (int rem = 0; rem < lowbitLen; ++rem) {
        vector<int> vec;
        for (int j = rem; j < n; j += lowbitLen) vec.pb(p[j]);
        sort(vec.begin(), vec.end());

        bool ok = true;
        int ptr = 0;
        for (int j = rem; j < n; j += lowbitLen) {
            if (vec[ptr++] != j) { ok = false; break; }
        }
        if (!ok) { printf("-1\n"); return 0; }

        for (int j = rem; j < n; j += lowbitLen) {
            if (p[j] != j) applySwap(j, p[j]);
        }
    }

    for (int i = 0; i < n; ++i) assert(p[i] == i);

    printf("%d\n", (int)answer.size());
    for (int x : answer) {
        if (x == 0) printf("0\n");
        else if (x < 0) printf("1 %d\n", -x);
        else printf("2 %d\n", x);
    }
    return 0;
}

SyntaxError: invalid syntax (3683182657.py, line 3)

## B 长跑

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

// 充电站结构体
struct Station {
    int p; // 位置
    int c; // 充电费用
};

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n, l, maxn, s;
    // 多组输入
    while (cin >> n >> l >> maxn >> s) {
        vector<Station> stations(n);
        for (int i = 0; i < n; i++) {
            cin >> stations[i].p >> stations[i].c;
        }

        // 按位置升序排序，便于后续判断
        sort(stations.begin(), stations.end(), [](const Station& a, const Station& b) {
            return a.p < b.p;
        });

        bool success = false;

        // 情况1：最大续航覆盖终点
        if (maxn >= l) {
            success = true;
        }

        // 情况2：使用单个充电站到达终点
        if (!success) {
            for (const auto& st : stations) {
                if (st.p <= maxn && st.p + maxn >= l && s >= st.c) {
                    success = true;
                    break;
                }
            }
        }

        // 情况3：使用两个充电站组合到达终点
        if (!success) {
            for (int i = 0; i < n; i++) {
                if (stations[i].p > maxn) continue; // 剪枝：第一个站不可达
                for (int j = i + 1; j < n; j++) {
                    if (stations[j].p - stations[i].p <= maxn && 
                        stations[j].p + maxn >= l &&
                        s >= stations[i].c + stations[j].c) {
                        success = true;
                        break;
                    }
                }
                if (success) break; // 找到可行解立即退出
            }
        }

        cout << (success ? "Yes" : "No") << "\n";
    }

    return 0;
}

## C 最长回文

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

// 全局变量存储 Manacher 半径
vector<int> p1, p2;

/**
 * @brief 将原字符串转换为 Manacher 算法适用的格式
 *        在每个字符之间插入 '#'，并在头部加 "$#" 防止越界
 * @param x 原字符串
 * @return 处理后的字符串
 */
string preprocessManacher(const string& x) {
    string temp = "$#";
    for (char ch : x) {
        temp += ch;
        temp += '#';
    }
    return temp;
}

/**
 * @brief Manacher 算法求最长回文半径数组
 * @param s 输入字符串（已预处理）
 * @param r 输出半径数组
 */
void manacher(const string& s, vector<int>& r) {
    int n = s.size();
    r.assign(n, 0);  // 初始化半径数组
    int center = 0;   // 当前回文中心

    for (int i = 0; i < n; ++i) {
        // 利用回文对称性优化初值
        if (center + r[center] > i)
            r[i] = min(r[2 * center - i], center + r[center] - i);

        // 向两侧扩展回文
        while (i - r[i] >= 0 && i + r[i] < n && s[i - r[i]] == s[i + r[i]])
            ++r[i];
        --r[i]; // 上面循环多加了一次

        // 更新当前回文最右边界中心
        if (i + r[i] > center + r[center])
            center = i;
    }
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;         // 读入字符串长度（未实际用到）
    string s, t;
    cin >> s >> t;    // 读入两个字符串

    // 预处理字符串
    s = preprocessManacher(s);
    t = preprocessManacher(t);

    // 计算两个字符串的回文半径数组
    manacher(s, p1);
    manacher(t, p2);

    int ans = 1;
    int len = t.size();

    // 遍历 t 中每个位置，寻找最大可扩展回文长度
    for (int i = 2; i < len; ++i) {
        int temp = max(p1[i], p2[i - 2]);
        while (i - temp >= 0 && i + temp - 2 < s.size() && s[i - temp] == t[i + temp - 2])
            ++temp;
        ans = max(ans, temp);
    }

    cout << ans - 1 << "\n";  // 输出最长公共回文长度
    return 0;
}

## D 优惠券

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

const int MAXN = 5e5 + 9;

int m;                       // 操作次数
int a;                       // 操作对象编号
int vis[MAXN];               // 物品状态：0 表示未在，1 表示在
int last_op[MAXN];           // 上一次操作的序号
string op;                   // 当前操作
set<int> pending_ops;        // 记录未知操作对应的序号

/**
 * @brief 处理一次操作序列
 */
void solve() {
    bool valid = true;       // 当前序列是否仍然有效
    int first_invalid = -1;  // 第一次非法操作序号
    memset(vis, 0, sizeof(vis));
    memset(last_op, 0, sizeof(last_op));
    pending_ops.clear();

    for (int i = 1; i <= m; ++i) {
        cin >> op;
        if (op == "I" || op == "O") {
            cin >> a;

            if (!valid) continue;  // 已出现非法操作，后续忽略

            // 更新物品状态
            if (op == "I") vis[a]++;
            else vis[a]--;

            // 检查是否非法
            if (vis[a] < 0 || vis[a] > 1) {
                auto it = pending_ops.lower_bound(last_op[a]);
                if (it == pending_ops.end()) {
                    // 找不到可匹配的未知操作 → 当前为第一次非法
                    first_invalid = i;
                    valid = false;
                } else {
                    // 有可匹配未知操作，进行修正
                    pending_ops.erase(it);
                    vis[a] = min(max(vis[a], 0), 1);  // 限制状态在 [0,1]
                }
            }

            last_op[a] = i;  // 记录本次操作序号
        } else {
            // 操作为 '?'，加入待匹配集合
            pending_ops.insert(i);
        }
    }

    cout << first_invalid << "\n";
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    while (cin >> m) {
        solve();
    }
    return 0;
}

## E 任意点

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

struct Point {
    int x, y;
};

int fa[105];        // 并查集父节点数组
Point a[105];       // 点坐标数组
int n;

/**
 * @brief 并查集查找操作（带路径压缩）
 * @param x 节点编号
 * @return 根节点编号
 */
int find(int x) {
    return fa[x] == x ? x : fa[x] = find(fa[x]);
}

/**
 * @brief 并查集合并操作，将 x 和 y 连通
 * @param x 节点编号
 * @param y 节点编号
 */
void merge(int x, int y) {
    fa[find(x)] = find(y);
}

/**
 * @brief 主逻辑函数
 */
void solve() {
    cin >> n;

    // 初始化并查集，每个点独立
    for (int i = 1; i <= n; ++i) fa[i] = i;

    // 输入所有点
    for (int i = 1; i <= n; ++i) {
        cin >> a[i].x >> a[i].y;
    }

    // 遍历每对点，如果 x 或 y 相同则连通
    for (int i = 1; i <= n; ++i) {
        for (int j = i + 1; j <= n; ++j) {
            if (a[i].x == a[j].x || a[i].y == a[j].y)
                merge(i, j);
        }
    }

    // 统计独立的连通区域
    int cnt = 0;
    for (int i = 1; i <= n; ++i) {
        if (find(i) == i) cnt++;
    }

    // 输出最少需要连接的操作次数
    cout << cnt - 1 << "\n";
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    solve();
    return 0;
}

## F 通配符匹配

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;
string a,b;
int n,la,lb;

bool dfs(int x,int y){
	if(x==0&&y==0) return 1;
	if(y==0){
		for(int i=x;i>0;i--){
			if(a[i]!='*') return 0;
		}
		return 1;
	}
	if(x==0) return 0;
	if(a[x]=='*'){
		for(int i=y;i>=0;i--){
			if(dfs(x-1,i)) return 1;
		}
		return 0;
	}
	if(a[x]==b[y]||a[x]=='?') return dfs(x-1,y-1);
	return 0;
}

void solve(){
	cin>>a>>n;
	a=' '+a,la=a.size()-1;
	while(n--){
		cin>>b;
		b=' '+b,lb=b.size()-1;
		if(dfs(la,lb)) cout<<"YES\n";
		else cout<<"NO\n";
	}
}
int main(){
	ios::sync_with_stdio(0);
	cin.tie(0),cout.tie(0);
	int T=1;
// 	cin>>T;
	while(T--) solve();
	return 0;
}
/*

*/


## G 汉诺塔

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    long long n;
    cin >> n;

    string s;
    long long a[10]; // 存储权值
    for (int i = 6; i >= 1; --i) {
        cin >> s;
        int idx = (s[0] - 'A') * 3 + (s[1] - 'A');
        a[idx] = i;
    }

    long long ans = 0;

    // 条件分支
    if ((a[1] > a[2] && a[3] > a[5]) || (a[1] <= a[2] && a[6] > a[7])) {
        ans = 2LL * pow(3LL, n - 1) - 1;
    } else if ((a[1] > a[2] && a[6] > a[7]) || (a[1] <= a[2] && a[3] > a[5])) {
        ans = pow(2LL, n) - 1;
    } else {
        ans = pow(3LL, n - 1);
    }

    cout << ans << "\n";
    return 0;
}

## H 马步距离

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

/**
 * @brief 计算骑士在无限棋盘上从 (0,0) 到 (dx,dy) 的最少步数
 * 公式来源：分析骑士移动的贪心规律 + 特殊情况处理
 */
int minKnightMoves(int dx, int dy) {
    dx = abs(dx);
    dy = abs(dy);

    // 利用对称性：保证 dx ≥ dy
    if (dx < dy) swap(dx, dy);

    // 特殊情况
    if (dx == 1 && dy == 0) return 3;
    if (dx == 2 && dy == 2) return 4;

    // 核心公式：
    // - dx + dy + steps 为偶数时可以完全到达
    int steps = max((dx + 1) / 2, (dx + dy + 2) / 3);

    // 调整步数使最终位置可到达
    if ((dx + dy + steps) % 2 != 0) steps++;

    return steps;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int x_start, y_start, x_target, y_target;
    cin >> x_start >> y_start >> x_target >> y_target;

    int dx = x_target - x_start;
    int dy = y_target - y_start;

    cout << minKnightMoves(dx, dy) << "\n";
    return 0;
}

## I 直方图最大矩形

In [ ]:
## add your code here
class Solution {
public:
    /**
     * @brief 计算柱状图中最大的矩形面积
     * @param heights 柱状图高度数组
     * @return 最大矩形面积
     */
    int largestRectangleArea(vector<int>& heights) {
        // 在头尾加入0，方便统一处理栈
        heights.insert(heights.begin(), 0);
        heights.push_back(0);

        int n = heights.size();
        stack<int> stk; // 存储柱子下标，保持单调递增
        int maxArea = 0;

        for (int i = 0; i < n; ++i) {
            // 当前柱子比栈顶柱子矮 → 计算矩形面积
            while (!stk.empty() && heights[i] < heights[stk.top()]) {
                int heightIndex = stk.top(); // 栈顶柱子高度
                stk.pop();

                int width = i - stk.top() - 1; // 宽度 = 当前索引 - 新栈顶 -1
                int area = heights[heightIndex] * width;
                maxArea = max(maxArea, area);
            }
            stk.push(i); // 入栈
        }

        return maxArea;
    }
};

## J 消防局的设立

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

const int N = 1010;

int parent[N];   // f[i] 原父节点
int depth[N];    // 节点深度
int dp[N];       // dp[i] 表示节点 i 的状态（类似距离或覆盖值）
int nodes[N];    // 节点编号，用于按深度排序

/**
 * @brief 按深度降序比较
 */
bool cmpByDepth(int x, int y) {
    return depth[x] > depth[y];
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    memset(dp, 0x3f, sizeof dp); // 初始化为大值（INF）
    parent[1] = 1;               // 根节点父节点自己

    // 读入父节点并计算深度
    depth[1] = 0;
    for (int i = 2; i <= n; ++i) {
        cin >> parent[i];
        depth[i] = depth[parent[i]] + 1;
        nodes[i] = i;
    }
    nodes[1] = 1;

    // 按深度从大到小处理
    sort(nodes + 1, nodes + n + 1, cmpByDepth);

    int ans = 0; // 需要的操作数量

    for (int i = 1; i <= n; ++i) {
        int v = nodes[i];
        int fa = parent[v];           // 父节点
        int grand = parent[fa];      // 祖父节点

        // 更新 dp[v]，表示最短距离或覆盖值
        dp[v] = min(dp[v], min(dp[fa] + 1, dp[grand] + 2));

        // 贪心选择：如果当前节点超过覆盖距离限制
        if (dp[v] > 2) {
            dp[grand] = 0;
            dp[parent[grand]] = min(dp[parent[grand]], 1);
            dp[parent[parent[grand]]] = min(dp[parent[parent[grand]]], 2);
            ans++; // 增加操作
        }
    }

    cout << ans << "\n";
    return 0;
}